# Terrain Diffusion Model — Training & Generation

Trains a single-channel (elevation-only) **DDPM diffusion model** on real terrain
data (Alps DEM, 256x256 patches) and generates new heightmaps for import into
**Unreal Engine 5**.

**Pipeline**
1. Build the denoising network (`UNet2DModel` from 🤗 `diffusers`).
2. Set up the noise scheduler (`DDPMScheduler`).
3. Load the 442 preprocessed patches via a `DataLoader`.
4. Train the model to predict the added noise — mixed precision, tuned for an 8 GB GPU.
5. Generate a single 256x256 heightmap via reverse diffusion.
6. Scale up to a large, seamless 512x512 map with **MultiDiffusion**.
7. Export both results as 16-bit PNGs ready for UE5's Landscape import tool.

**Core contribution:** running diffusion training on a consumer 8 GB laptop GPU.
The memory bottleneck is solved with `bfloat16` mixed precision + a small batch
size, giving a ~38x training speed-up over full precision.

**References:** DDPM formulation (Ho et al., 2020), the *Terrain Diffusion*
approach (Goslin, 2025), and MultiDiffusion (Bar-Tal et al., 2023) for the
large-map generation step.

## 1. Imports & device check

Everything the notebook needs, in one place.

In [ ]:
import glob
import time

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from scipy.ndimage import gaussian_filter
from torch.utils.data import Dataset, DataLoader
from diffusers import UNet2DModel, DDPMScheduler

# All training/inference runs on GPU — CPU is not practical for diffusion models.
DEVICE = "cuda"
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

## 2. Model — the denoising U-Net

A `UNet2DModel` takes a **noisy patch + a timestep** and predicts the noise it
contains. Input/output are single-channel (elevation only, no RGB).

- `block_out_channels` sets the feature-map capacity at each resolution level
  (64 → 128 → 256 → 512 as the network downsamples).
- Attention is used only at the deepest (lowest-resolution) level, where it's
  cheapest, to help the model relate distant regions of the patch — e.g. keeping
  an extended mountain ridge coherent across the whole tile.

In [ ]:
model = UNet2DModel(
    sample_size=256,        # patch resolution (256x256)
    in_channels=1,           # single channel: elevation only
    out_channels=1,          # model predicts the noise, same shape as input
    layers_per_block=2,      # residual blocks per resolution level
    block_out_channels=(64, 128, 256, 512),   # feature capacity per level (shallow -> deep)
    down_block_types=("DownBlock2D", "DownBlock2D", "DownBlock2D", "AttnDownBlock2D"),
    up_block_types=("AttnUpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D"),
)
model = model.to(DEVICE)

# ~61.8M trainable parameters — a deliberate balance between learning capacity
# and what an 8 GB GPU can hold in memory alongside activations and gradients.
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params/1e6:.1f}M")

## 3. Noise scheduler

`DDPMScheduler` defines the forward (noising) process over 1000 timesteps —
i.e. how much Gaussian noise is mixed into a clean patch at each step.

- **Training:** `add_noise(clean, noise, t)` — corrupt a clean patch to a random
  timestep `t` so the model has something to learn to reverse.
- **Generation:** `step(...)` — run this process backwards, removing a little
  noise at a time until only terrain remains.

In [ ]:
scheduler = DDPMScheduler(num_train_timesteps=1000)
print("Training timesteps:", scheduler.config.num_train_timesteps)

## 4. Data — Dataset & DataLoader

Loads the 442 preprocessed, normalized 256x256 elevation patches (values
rescaled to `[-1, 1]`, see the preprocessing notebook). Each patch is stored as
a `.npy` file for fast loading.

`batch_size=4` is the largest batch that reliably fits in 8 GB of VRAM at this
resolution — see the memory/efficiency discussion in the training loop below.

In [ ]:
class TerrainDataset(Dataset):
    """Loads preprocessed 256x256 elevation patches as single-channel tensors."""

    def __init__(self, patch_dir):
        # Sorted for reproducible ordering; DataLoader shuffling handles randomization.
        self.files = sorted(glob.glob(patch_dir + "/*.npy"))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        patch = np.load(self.files[idx])            # (256, 256) NumPy array
        return torch.from_numpy(patch).unsqueeze(0)  # -> (1, 256, 256) tensor, channel-first


ds = TerrainDataset("../outputs/processed/patches")
loader = DataLoader(ds, batch_size=4, shuffle=True)
print("Patches:", len(ds), "| Batches per epoch:", len(loader))

## 5. Optimizer

`AdamW` updates the model's weights during training. Instantiated **after**
the model cell so it's bound to the current parameters — re-run this cell if
the model is ever rebuilt from scratch.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
print("Optimizer ready.")

## 6. Training loop

For every batch, each step:
1. Sample random Gaussian noise and a random timestep `t` per patch.
2. Corrupt the clean patch to that timestep (`add_noise`).
3. Ask the model to predict the noise that was added.
4. Minimize the MSE between the predicted and true noise.

**Efficiency — the core contribution of this project:** training at full batch
size and full precision overflowed the 8 GB GPU's memory. This is resolved with:
- **`bfloat16` mixed precision** (`torch.autocast`) — halves memory use and
  speeds up the matrix multiplications that dominate training.
- **A small batch size (4)** — tuned to the remaining memory budget.

Together these cut the time per training step from ~32s to under 1s (~38x
speed-up), making training on consumer hardware practical.

Checkpoints save every 10 epochs so training can resume after an interruption
without losing progress.

In [ ]:
num_epochs = 50
save_path = "../outputs/terrain_model"

torch.cuda.empty_cache()   # start with a clean memory slate
model.train()

for epoch in range(num_epochs):
    epoch_loss = 0.0
    t0 = time.time()

    for batch in loader:
        batch = batch.to(DEVICE)

        # --- Forward diffusion: corrupt a clean patch with known noise ---
        noise = torch.randn_like(batch)
        t = torch.randint(0, scheduler.config.num_train_timesteps,
                          (batch.shape[0],), device=DEVICE)
        noisy = scheduler.add_noise(batch, noise, t)

        # --- Predict the noise, in bfloat16 mixed precision (memory/speed win) ---
        with torch.autocast("cuda", dtype=torch.bfloat16):
            pred = model(noisy, t).sample
            loss = F.mse_loss(pred, noise)   # how far off the noise prediction was

        # --- Backpropagation & weight update ---
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()

    avg = epoch_loss / len(loader)
    print(f"epoch {epoch+1:3d}/{num_epochs} | avg loss {avg:.4f} | {time.time()-t0:.0f}s")

    # Periodic checkpoint — safe to interrupt training and resume from here.
    if (epoch + 1) % 10 == 0:
        model.save_pretrained(save_path)

model.save_pretrained(save_path)
print("Training done. Model saved ->", save_path)

### Reload the trained model (optional)

Only needed after a kernel restart, to skip re-training: loads the saved
checkpoint back onto the GPU in evaluation mode, ready for generation.

In [ ]:
# model = UNet2DModel.from_pretrained(save_path).to(DEVICE)
# model.eval()
# print("Trained model reloaded.")

## 7. Generate a single heightmap patch

**Reverse diffusion:** start from pure random noise and let the trained model
remove it step by step (1000 steps) until a coherent 256x256 terrain patch
emerges. This is the same mechanism as training, run backwards with no ground
truth — purely the model's learned denoising.

In [ ]:
model.eval()  # disable dropout/batchnorm training behavior for inference
sample = torch.randn(1, 1, 256, 256, device=DEVICE)   # start: pure Gaussian noise

scheduler.set_timesteps(1000)
for t in scheduler.timesteps:
    with torch.no_grad():                      # no gradients needed at inference
        noise_pred = model(sample, t).sample    # model's guess at the noise present
    sample = scheduler.step(noise_pred, t, sample).prev_sample   # remove a bit of it

generated = sample[0, 0].cpu().float().numpy()  # back to a plain (256, 256) array

plt.figure(figsize=(6, 6))
plt.imshow(generated, cmap="terrain")
plt.colorbar(label="elevation (normalized)")
plt.title("Generated terrain (trained model)")
plt.axis("off")
plt.show()
print("Value range:", round(float(generated.min()), 3), "to", round(float(generated.max()), 3))

## 8. Export the single patch for Unreal Engine 5

UE5's Landscape import tool expects a **16-bit grayscale PNG** heightmap. A
light Gaussian blur removes single-pixel noise spikes that would otherwise
read as sharp, unnatural needles once rendered in 3D.

In [ ]:
g = generated.astype(np.float32)
g = gaussian_filter(g, sigma=2)                 # smooth high-frequency spikes
g = (g - g.min()) / (g.max() - g.min())         # rescale to [0, 1]
g16 = (g * 65535).astype(np.uint16)             # convert to 16-bit for UE5 import

imageio.imwrite("../outputs/heightmap_smooth.png", g16)
print("Saved -> ../outputs/heightmap_smooth.png", g16.shape, g16.dtype)

## 9. MultiDiffusion — scaling beyond a single patch

A single 256x256 patch is too small for a usable game map. **MultiDiffusion**
(Bar-Tal et al., 2023) lifts this limit by running several overlapping
256x256 "windows" over one large canvas and blending their predictions at
**every** denoising step — not just at the end. Averaging the overlaps at
each step keeps neighboring windows consistent with each other as they
denoise together, so the seams between them disappear.

In [ ]:
@torch.no_grad()
def generate_multidiffusion(out_size=512, patch=256, stride=128, steps=1000):
    """
    Generate a large, seamless heightmap using MultiDiffusion.
    Blends overlapping patches DURING each denoising step for global coherence.
    """
    # 1. big canvas of pure noise
    canvas = torch.randn(1, 1, out_size, out_size, device=DEVICE)

    # 2. top-left coords of every overlapping tile
    coords = []
    for y in range(0, out_size - patch + 1, stride):
        for x in range(0, out_size - patch + 1, stride):
            coords.append((y, x))

    scheduler.set_timesteps(steps)
    for t in scheduler.timesteps:
        acc = torch.zeros_like(canvas)      # accumulates denoised predictions
        cnt = torch.zeros_like(canvas)      # counts overlaps per pixel
        for (y, x) in coords:
            tile = canvas[:, :, y:y+patch, x:x+patch]        # current tile
            noise_pred = model(tile, t).sample                # predict noise
            denoised = scheduler.step(noise_pred, t, tile).prev_sample
            acc[:, :, y:y+patch, x:x+patch] += denoised       # add back
            cnt[:, :, y:y+patch, x:x+patch] += 1
        canvas = acc / cnt                  # 3. average overlaps -> seamless

    return canvas[0, 0].cpu().float().numpy()

### Generate and preview the 512x512 map

Runs the function above at `out_size=512` (a 2x2 grid of overlapping 256x256
windows) and displays the result.

In [ ]:
big = generate_multidiffusion(out_size=512)

plt.figure(figsize=(7, 7))
plt.imshow(big, cmap="terrain")
plt.axis("off")
plt.title("MultiDiffusion 512x512")
plt.show()
print("Value range:", round(float(big.min()), 3), "to", round(float(big.max()), 3))

### Export the large map for Unreal Engine 5

Same 16-bit PNG export as before, with a lighter blur (`sigma=1.5`) since the
larger canvas already reads smoother than a single small patch.

In [ ]:
g = gaussian_filter(big.astype(np.float32), sigma=1.5)   # light smoothing
g = (g - g.min()) / (g.max() - g.min())                   # rescale to [0, 1]
imageio.imwrite("../outputs/heightmap_multidiff_512.png", (g * 65535).astype(np.uint16))
print("Saved -> ../outputs/heightmap_multidiff_512.png")